In [ ]:
!pip install -q sentence-transformers #Creates Embeddings
!pip install -q faiss-cpu #Vector DB
!pip install -q transformers #Load LLM
!pip install -q accelerate
!pip install -q pypdf #Read pdfs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.2/332.2 kB 8.6 MB/s eta 0:00:00


In [ ]:
import os
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

import numpy as np
import faiss

from pypdf import PdfReader

from sentence_transformers import SentenceTransformer
from transformers import pipeline

In [ ]:
pdf_files = [
    "DL research.pdf",
    "SQL.pdf",
    "EDA on SQL .pdf",
    "sql interview question.pdf",
    "PBI questions.pdf",
    "stats..pdf"
]

text = ""

for file in pdf_files:

    reader = PdfReader(file)

    for page in reader.pages:

        page_text = page.extract_text()

        if page_text:
            text += page_text

In [ ]:
chunk_size = 500
chunks = []

for i in range(0, len(text), chunk_size):

    chunks.append(text[i:i+chunk_size])

print("Total chunks:", len(chunks))

Total chunks: 1020


In [ ]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(chunks)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("Vector database ready")

Vector database ready


# LLM

## TinyLlama

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_name_tiny = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer_tiny = AutoTokenizer.from_pretrained(model_name_tiny)

model_tiny = AutoModelForCausalLM.from_pretrained(
    model_name_tiny,
    device_map="auto",
    torch_dtype="auto"
)

generator_tiny = pipeline(
    "text-generation",
    model=model_tiny,
    tokenizer=tokenizer_tiny
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## Phi 2

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, pipeline

model_name = "microsoft/phi-2"

config = AutoConfig.from_pretrained(model_name)

config.pad_token_id = config.eos_token_id

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    config=config,
    device_map="auto",
    torch_dtype="auto"
)

generator_phi = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    max_length = None
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
def retrieve_context(question):

    question_embedding = embedding_model.encode([question])

    distances, indices = index.search(
        np.array(question_embedding), k=3
    )

    results = [chunks[i] for i in indices[0]]

    return " ".join(results)

In [ ]:
def ask_tiny(question):

    context = retrieve_context(question)

    messages = [
        {
            "role": "system",
            "content": "You are a helpful AI assistant. Answer the question briefly in 2-3 sentences."
        },
        {
            "role": "user",
            "content": f"Question: {question}\n\nContext: {context}"
        }
    ]

    prompt = tokenizer_tiny.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    result = generator_tiny(
        prompt,
        max_new_tokens=80,
        do_sample=False,
        return_full_text=False
    )

    return result[0]["generated_text"].strip()

In [ ]:
def ask_phi(question):

    context = retrieve_context(question)

    prompt = f"""
Answer the question in 3 sentences.

Question: {question}

Context: {context}

Answer:
"""

    result = generator_phi(
        prompt,
        max_new_tokens=120,
        do_sample=False,
        return_full_text=False
    )

    return result[0]["generated_text"]

In [ ]:
def score_models(phi_answer, tiny_answer, context):

    phi_vec = embedding_model.encode(phi_answer)
    tiny_vec = embedding_model.encode(tiny_answer)
    context_vec = embedding_model.encode(context)

    # cosine similarity
    phi_sim = np.dot(phi_vec, context_vec) / (
        np.linalg.norm(phi_vec) * np.linalg.norm(context_vec)
    )

    tiny_sim = np.dot(tiny_vec, context_vec) / (
        np.linalg.norm(tiny_vec) * np.linalg.norm(context_vec)
    )

    # normalize to 0–1
    phi_score = (phi_sim + 1) / 2
    tiny_score = (tiny_sim + 1) / 2

    return phi_score, tiny_score

In [ ]:
def compare_models(question):

    # Get answers from both models
    phi_answer = ask_phi(question)
    tiny_answer = ask_tiny(question)

    # Retrieve context
    contexts = retrieve_context(question)

    # Score answers against context
    phi_score, tiny_score = score_models(phi_answer, tiny_answer, contexts[0])

    # Decide final answer
    if phi_score > tiny_score:
        final_answer = phi_answer
        winner = "Phi-2"
    else:
        final_answer = tiny_answer
        winner = "TinyLlama"

    print("\nQuestion:")
    print(question)

    print("\nPhi-2 Answer:")
    print(phi_answer)

    print("\nTinyLlama Answer:")
    print(tiny_answer)

    print("\nFinal Answer Selected:")
    print(final_answer)

    print("\nWinning Model:", winner)

    print("\nModel Scores (0–1 scale)")
    print("Phi-2 Score:", round(phi_score,2))
    print("TinyLlama Score:", round(tiny_score,2))

In [ ]:
compare_models("What is statistics?")


Question:
What is statistics?

Phi-2 Answer:

Statistics is the science of collecting, organizing, and analyzing data. It is used to make conclusions about the population by using analytical tools on the sample. Descriptive statistics are used to organize and summarize data, while inferential statistics are used to make conclusions about the population. Measures of central tendency, such as mean, median, and mode, are used to describe the center of a dataset. Measures of dispersion, such as range, variance, and standard deviation, are used to describe the spread of a dataset. Different types of distributions, such as Bernoulli, uniform, binomial

TinyLlama Answer:
Statistics is the science of collecting, organizing, and analyzing data to gain insights and make informed decisions. Data is defined as facts or pieces of information, and statistics are the science of collecting, organizing, and analyzing data to provide insights and make decisions. Data is collected from various sources, 

In [ ]:
# def chatbot():

#     print("RAG Chatbot Started")
#     print("Type 'exit' to stop\n")

#     while True:

#         question = input("\nAsk a question: ")

#         if question.lower() == "exit":
#             print("Chat ended.")
#             break

#         print("\nChoose model:")
#         print("1 - Phi-2")
#         print("2 - TinyLlama")
#         print("3 - Compare both")

#         choice = input("Enter choice: ")

#         if choice == "1":

#             answer = ask_phi(question)

#             print("\nPhi-2 Answer:")
#             print(answer)

#         elif choice == "2":

#             answer = ask_tiny(question)

#             print("\nTinyLlama Answer:")
#             print(answer)

#         elif choice == "3":

#             compare_models(question)

#         else:

#             print("Invalid option. Try again.")


# # START CHATBOT
# chatbot()

# Fine Tuning

In [ ]:
!pip install -q datasets peft bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 20.9 MB/s eta 0:00:00


In [ ]:
training_data = []

for chunk in chunks[:40]:   # reduce samples to avoid OOM
    training_data.append({
        "instruction": "Explain the following concept clearly.",
        "input": chunk,
        "output": chunk
    })

print("Training samples:", len(training_data))

Training samples: 40


In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(training_data)

In [ ]:
def format_prompt(example):

    prompt = f"""
Instruction: {example['instruction']}

Input:
{example['input']}

Response:
{example['output']}
"""

    return {"text": prompt}

dataset = dataset.map(format_prompt)

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

In [ ]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True
)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name_tiny = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer_tiny = AutoTokenizer.from_pretrained(model_name_tiny)

model_tiny = AutoModelForCausalLM.from_pretrained(
    model_name_tiny,
    device_map="auto",
    quantization_config=bnb_config
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
from transformers import AutoConfig

model_name_phi = "microsoft/phi-2"

config_phi = AutoConfig.from_pretrained(model_name_phi)
config_phi.pad_token_id = config_phi.eos_token_id

tokenizer_phi = AutoTokenizer.from_pretrained(model_name_phi)
tokenizer_phi.pad_token = tokenizer_phi.eos_token

model_phi = AutoModelForCausalLM.from_pretrained(
    model_name_phi,
    config=config_phi,
    device_map="auto",
    quantization_config=bnb_config
)

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

In [ ]:
def tokenize_tiny(example):

    return tokenizer_tiny(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized_dataset_tiny = dataset.map(tokenize_tiny, batched=True)

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

In [ ]:
def tokenize_phi(example):

    return tokenizer_phi(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized_dataset_phi = dataset.map(tokenize_phi, batched=True)

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model_tiny = get_peft_model(model_tiny, lora_config)
model_phi = get_peft_model(model_phi, lora_config)

model_tiny.print_trainable_parameters()
model_phi.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:78: UserWarning: The PEFT config's `base_model_name_or_path` was renamed from 'TinyLlama/TinyLlama-1.1B-Chat-v1.0' to 'microsoft/phi-2'. Please ensure that the correct base model is loaded when loading this checkpoint.
  warnings.warn(


trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023
trainable params: 2,621,440 || all params: 2,782,305,280 || trainable%: 0.0942


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    logging_steps=10
)

In [ ]:
from trl import SFTTrainer

trainer_tiny = SFTTrainer(
    model=model_tiny,
    train_dataset=tokenized_dataset_tiny,
    args=training_args
)

trainer_tiny.train()

Truncating train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

{'loss': '3.368', 'grad_norm': '15.81', 'learning_rate': '2.75e-05', 'entropy': '2.324', 'num_tokens': '1.024e+04', 'mean_token_accuracy': '0.5036', 'epoch': '0.5'}
{'loss': '2.759', 'grad_norm': '13.25', 'learning_rate': '2.5e-06', 'entropy': '2.321', 'num_tokens': '2.048e+04', 'mean_token_accuracy': '0.5862', 'epoch': '1'}


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


{'train_runtime': '16.31', 'train_samples_per_second': '2.452', 'train_steps_per_second': '1.226', 'train_loss': '3.064', 'epoch': '1'}


TrainOutput(global_step=20, training_loss=3.0638577461242678, metrics={'train_runtime': 16.3104, 'train_samples_per_second': 2.452, 'train_steps_per_second': 1.226, 'total_flos': 127259293777920.0, 'train_loss': 3.0638577461242678})

In [ ]:
trainer_tiny.model.save_pretrained("tinyllama-finetuned")
tokenizer_tiny.save_pretrained("tinyllama-finetuned")

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


('tinyllama-finetuned/tokenizer_config.json',
 'tinyllama-finetuned/chat_template.jinja',
 'tinyllama-finetuned/tokenizer.json')

In [ ]:
trainer_phi = SFTTrainer(
    model=model_phi,
    train_dataset=tokenized_dataset_phi,
    args=training_args
)

trainer_phi.train()

Truncating train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

{'loss': '4.427', 'grad_norm': '25.88', 'learning_rate': '2.75e-05', 'entropy': '2.455', 'num_tokens': '1.024e+04', 'mean_token_accuracy': '0.4399', 'epoch': '0.5'}
{'loss': '4.322', 'grad_norm': '26', 'learning_rate': '2.5e-06', 'entropy': '2.38', 'num_tokens': '2.048e+04', 'mean_token_accuracy': '0.4363', 'epoch': '1'}
{'train_runtime': '18.07', 'train_samples_per_second': '2.214', 'train_steps_per_second': '1.107', 'train_loss': '4.375', 'epoch': '1'}


TrainOutput(global_step=20, training_loss=4.374613189697266, metrics={'train_runtime': 18.0658, 'train_samples_per_second': 2.214, 'train_steps_per_second': 1.107, 'total_flos': 325783545446400.0, 'train_loss': 4.374613189697266})

In [ ]:
trainer_phi.model.save_pretrained("phi2-finetuned")
tokenizer_phi.save_pretrained("phi2-finetuned")

('phi2-finetuned/tokenizer_config.json', 'phi2-finetuned/tokenizer.json')

In [ ]:
from peft import PeftModel
from transformers import pipeline

# TinyLlama
base_tiny = AutoModelForCausalLM.from_pretrained(
    model_name_tiny,
    device_map="auto",
    quantization_config=bnb_config
)

model_tiny = PeftModel.from_pretrained(base_tiny, "tinyllama-finetuned")

generator_tiny = pipeline(
    "text-generation",
    model=model_tiny,
    tokenizer=tokenizer_tiny,
    max_new_tokens=200
)

# Phi-2
base_phi = AutoModelForCausalLM.from_pretrained(
    model_name_phi,
    config=config_phi,
    device_map="auto",
    quantization_config=bnb_config
)

model_phi = PeftModel.from_pretrained(base_phi, "phi2-finetuned")

generator_phi = pipeline(
    "text-generation",
    model=model_phi,
    tokenizer=tokenizer_phi,
    max_new_tokens=200
)

print("Fine-tuned models loaded successfully")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

Fine-tuned models loaded successfully


In [ ]:
def score_models(phi_answer, tiny_answer, context):

    # Encode text
    phi_vec = embedding_model.encode(phi_answer)
    tiny_vec = embedding_model.encode(tiny_answer)
    context_vec = embedding_model.encode(context)

    # ---------------------------
    # 1️⃣ Context Relevance
    # ---------------------------

    phi_relevance = np.dot(phi_vec, context_vec) / (
        np.linalg.norm(phi_vec) * np.linalg.norm(context_vec)
    )

    tiny_relevance = np.dot(tiny_vec, context_vec) / (
        np.linalg.norm(tiny_vec) * np.linalg.norm(context_vec)
    )

    # ---------------------------
    # 2️⃣ Semantic Similarity
    # ---------------------------

    phi_similarity = phi_relevance
    tiny_similarity = tiny_relevance

    # ---------------------------
    # 3️⃣ Answer Completeness
    # ---------------------------

    phi_length = len(phi_answer.split())
    tiny_length = len(tiny_answer.split())

    phi_completeness = min(phi_length / 50, 1)
    tiny_completeness = min(tiny_length / 50, 1)

    # ---------------------------
    # Final Score
    # ---------------------------

    phi_score = (
        0.4 * phi_relevance +
        0.4 * phi_similarity +
        0.2 * phi_completeness
    )

    tiny_score = (
        0.4 * tiny_relevance +
        0.4 * tiny_similarity +
        0.2 * tiny_completeness
    )

    return {
        "phi": {
            "relevance": phi_relevance,
            "similarity": phi_similarity,
            "completeness": phi_completeness,
            "final": phi_score
        },
        "tiny": {
            "relevance": tiny_relevance,
            "similarity": tiny_similarity,
            "completeness": tiny_completeness,
            "final": tiny_score
        }
    }

In [ ]:
def compare_models(question):

    # Generate answers
    phi_answer = ask_phi(question)
    tiny_answer = ask_tiny(question)

    # Retrieve context (used internally for scoring only)
    context = retrieve_context(question)

    # Score answers
    scores = score_models(phi_answer, tiny_answer, context)

    phi_score = scores["phi"]["final"]
    tiny_score = scores["tiny"]["final"]

    # Decide winner
    if phi_score > tiny_score:
        winner = "Fine-tuned Phi-2"
        final_answer = phi_answer
    else:
        winner = "Fine-tuned TinyLlama"
        final_answer = tiny_answer

    print("\n==============================")
    print("Question:")
    print(question)

    print("\nFine-tuned Phi-2 Answer:")
    print(phi_answer)

    print("\nFine-tuned TinyLlama Answer:")
    print(tiny_answer)

    print("\n------------------------------")

    print("\nScores")

    print("\nPhi-2")
    print("Relevance:", round(scores["phi"]["relevance"],2))
    print("Similarity:", round(scores["phi"]["similarity"],2))
    print("Completeness:", round(scores["phi"]["completeness"],2))
    print("Final Score:", round(scores["phi"]["final"],2))

    print("\nTinyLlama")
    print("Relevance:", round(scores["tiny"]["relevance"],2))
    print("Similarity:", round(scores["tiny"]["similarity"],2))
    print("Completeness:", round(scores["tiny"]["completeness"],2))
    print("Final Score:", round(scores["tiny"]["final"],2))

    print("\n==============================")

    print("\nWinning Model:", winner)

    print("\nFinal Selected Answer:")
    print(final_answer)

In [ ]:
def chatbot():

    print("Fine-Tuned RAG Chatbot Started")
    print("Type 'exit' to stop\n")

    while True:

        question = input("\nAsk a question: ")

        if question.lower() == "exit":
            print("Chat ended.")
            break

        print("\nChoose option:")
        print("1 - Fine-tuned Phi-2")
        print("2 - Fine-tuned TinyLlama")
        print("3 - Compare both")

        choice = input("Enter choice: ")

        if choice == "1":

            answer = ask_phi(question)

            print("\nPhi-2 Answer:")
            print(answer)

        elif choice == "2":

            answer = ask_tiny(question)

            print("\nTinyLlama Answer:")
            print(answer)

        elif choice == "3":

            compare_models(question)

        else:

            print("Invalid option")

chatbot()

Fine-Tuned RAG Chatbot Started
Type 'exit' to stop


Ask a question: what is deep learning?

Choose option:
1 - Fine-tuned Phi-2
2 - Fine-tuned TinyLlama
3 - Compare both
Enter choice: 3

Question:
what is deep learning?

Fine-tuned Phi-2 Answer:

Deep learning is a type of machine learning that uses neural networks to learn from data. It is motivated by the success of gradient descent applied to differentiable feedforward networks for classification. In supervised learning, deep feedforward networks trained with gradient-based learning seem practically guaranteed to succeed with enough hidden units and training data. Can this same recipe be applied to generative modeling?

Deep convolutional networks usually require pooling operations to decrease the spatial size of each successive layer. Feedforward co-learning, the RBM is relatively straightforward to train because we can compute P(h|v

Fine-tuned TinyLlama Answer:
Deep learning is a technique for training artificial neural networks